In [125]:
import asyncio
import json
import websockets
from datetime import datetime

In [126]:
buy_volume = 0
sell_volume = 0

trade_count = 0

last_price = 0

delta = 0
cvd = 0

prev_buy_volume = 0
prev_sell_volume = 0

interval_buy_volume = 0
interval_sell_volume = 0

buy_sell_ratio = 0

bid_volume = 0
ask_volume = 0

orderbook_imbalance = 0

In [127]:
from collections import deque

whale_threshold = 100000

whale_buys = 0
whale_sells = 0

whale_buy_value = 0
whale_sell_value = 0

recent_whales = deque(maxlen=50)

In [128]:
absorption_score = 0

last_price_snapshot = 0

cluster_score = 0

In [129]:
market_state = "NEUTRAL"

bullish_absorption_score = 0
bearish_absorption_score = 0

In [130]:
long_score = 0
short_score = 0

trade_signal = "NO TRADE"

long_confidence = 0
short_confidence = 0

In [131]:
symbol = "btcusdt"

trade_url = f"wss://fstream.binance.com/ws/{symbol}@trade"

In [132]:
depth_url = (
    f"wss://fstream.binance.com/ws/{symbol}@depth20@100ms"
)

In [133]:
async def trade_stream():

    global buy_volume
    global sell_volume
    global trade_count
    global last_price

    global whale_buys
    global whale_sells
    global whale_buy_value
    global whale_sell_value
    global recent_whales

    async with websockets.connect(trade_url) as ws:

        print("Connected to Binance")

        while True:

            msg = json.loads(await ws.recv())

            price = float(msg["p"])
            qty = float(msg["q"])

            value = price * qty

            if value >= whale_threshold:

                recent_whales.append({
                    "side": "SELL" if msg["m"] else "BUY",
                    "value": value,
                    "price": price
                })
            
                if msg["m"]:
                    whale_sells += 1
                    whale_sell_value += value
                else:
                    whale_buys += 1
                    whale_buy_value += value

            last_price = price

            trade_count += 1

            if msg["m"]:
                sell_volume += value
            else:
                buy_volume += value

In [134]:
async def delta_engine():

    global delta
    global cvd

    global prev_buy_volume
    global prev_sell_volume

    global interval_buy_volume
    global interval_sell_volume

    while True:

        interval_buy_volume = buy_volume - prev_buy_volume
        interval_sell_volume = sell_volume - prev_sell_volume

        delta = interval_buy_volume - interval_sell_volume

        cvd += delta

        prev_buy_volume = buy_volume
        prev_sell_volume = sell_volume

        if interval_sell_volume > 0:
            buy_sell_ratio = interval_buy_volume / interval_sell_volume
        else:
            buy_sell_ratio = 999

        await asyncio.sleep(5)

In [135]:
async def orderbook_stream():

    global bid_volume
    global ask_volume
    global orderbook_imbalance

    async with websockets.connect(depth_url) as ws:

        print("📚 Order Book Connected")

        while True:

            msg = json.loads(await ws.recv())

            bids = msg["b"]
            asks = msg["a"]

            bid_volume = sum(
                float(level[1])
                for level in bids
            )

            ask_volume = sum(
                float(level[1])
                for level in asks
            )

            total = bid_volume + ask_volume

            if total > 0:
                orderbook_imbalance = (
                    bid_volume - ask_volume
                ) / total

In [136]:
async def absorption_engine():

    global absorption_score
    global last_price_snapshot

    while True:

        price_change = abs(
            last_price - last_price_snapshot
        )

        volume_total = (
            interval_buy_volume +
            interval_sell_volume
        )

        absorption_score = 0

        if volume_total > 0:

            # large volume
            if volume_total > 500000:

                # small movement
                if price_change < 50:

                    absorption_score = min(
                        100,
                        int(
                            volume_total / 10000
                        )
                    )

        last_price_snapshot = last_price

        await asyncio.sleep(5)

In [137]:
async def whale_cluster_engine():

    global cluster_score

    while True:

        cluster_score = len(recent_whales)

        await asyncio.sleep(5)

In [138]:
async def market_state_engine():

    global market_state

    global bullish_absorption_score
    global bearish_absorption_score

    last_cvd = 0
    last_price = 0

    while True:

        price_change = globals()["last_price"] - last_price

        cvd_change = cvd - last_cvd

        bullish_absorption_score = 0
        bearish_absorption_score = 0

        # -------------------------
        # BULLISH ABSORPTION
        # -------------------------

        if price_change < 0:

            if delta < 0:

                if orderbook_imbalance > 0:

                    if whale_buy_value > whale_sell_value:

                        bullish_absorption_score += 25

                    if cvd_change < 0:

                        bullish_absorption_score += 25

                    if abs(price_change) < 50:

                        bullish_absorption_score += 25

                    if buy_sell_ratio > 0.8:

                        bullish_absorption_score += 25

        # -------------------------
        # BEARISH ABSORPTION
        # -------------------------

        if price_change > 0:

            if delta > 0:

                if orderbook_imbalance < 0:

                    if whale_sell_value > whale_buy_value:

                        bearish_absorption_score += 25

                    if cvd_change > 0:

                        bearish_absorption_score += 25

                    if abs(price_change) < 50:

                        bearish_absorption_score += 25

                    if buy_sell_ratio < 1.2:

                        bearish_absorption_score += 25

        # -------------------------
        # MARKET STATES
        # -------------------------

        market_state = "NEUTRAL"

        if bullish_absorption_score >= 50:
            market_state = "BULLISH ABSORPTION"

        elif bearish_absorption_score >= 50:
            market_state = "BEARISH ABSORPTION"

        elif (
            delta > 0
            and cvd_change > 0
            and orderbook_imbalance > 0
        ):
            market_state = "BULL TREND"

        elif (
            delta < 0
            and cvd_change < 0
            and orderbook_imbalance < 0
        ):
            market_state = "BEAR TREND"

        elif (
            abs(delta) < 100000
            and abs(orderbook_imbalance) < 0.05
        ):
            market_state = "CONSOLIDATION"

        last_price = globals()["last_price"]
        last_cvd = cvd

        await asyncio.sleep(5)

In [139]:
async def sniper_engine():

    global long_score
    global short_score
    global trade_signal

    global long_confidence
    global short_confidence

    while True:

        long_score = 0
        short_score = 0

        # -------------------
        # DELTA
        # -------------------

        if delta > 0:
            long_score += 20
        else:
            short_score += 20

        # -------------------
        # BUY/SELL RATIO
        # -------------------

        if buy_sell_ratio > 1:
            long_score += 10
        else:
            short_score += 10

        # -------------------
        # ORDER BOOK
        # -------------------

        if orderbook_imbalance > 0:
            long_score += 20
        else:
            short_score += 20

        # -------------------
        # WHALES
        # -------------------

        if whale_buy_value > whale_sell_value:
            long_score += 20
        else:
            short_score += 20

        # -------------------
        # ABSORPTION
        # -------------------

        if bullish_absorption_score > bearish_absorption_score:
            long_score += 15

        if bearish_absorption_score > bullish_absorption_score:
            short_score += 15

        # -------------------
        # CVD MOMENTUM
        # -------------------

        if cvd > 0:
            long_score += 15
        else:
            short_score += 15

        # -------------------
        # SIGNAL
        # -------------------

        trade_signal = "NO TRADE"

        if long_score >= 70:
            trade_signal = "LONG SETUP"

        elif short_score >= 70:
            trade_signal = "SHORT SETUP"

        total = long_score + short_score

        if total > 0:
            long_confidence = round(
                long_score / total * 100,
                2
            )
        
            short_confidence = round(
                short_score / total * 100,
                2
            )

        await asyncio.sleep(5)

In [140]:
async def dashboard():

    while True:

        print("\n" + "="*40)
        print("PHASE 1")
        print("="*40)
        
        print(f"Price       : {last_price:,.2f}")
        print(f"Buy Volume  : {buy_volume:,.2f}")
        print(f"Sell Volume : {sell_volume:,.2f}")
        print(f"Trades      : {trade_count}")

        
        
        print("\n" + "="*40)
        print("PHASE 2")
        print("="*40)

        print(f"Interval Buy Volume  : {interval_buy_volume:,.2f}")
        print(f"Interval Sell Volume : {interval_sell_volume:,.2f}")

        print(f"Buy/Sell Ratio       : {buy_sell_ratio:.2f}")
        
        print(f"Delta       : {delta:,.2f}")
        print(f"CVD         : {cvd:,.2f}")

        print("\n" + "="*40)
        print("PHASE 3")
        print("="*40)
        
        print(f"Bid Liquidity         : {bid_volume:,.2f}")
        print(f"Ask Liquidity         : {ask_volume:,.2f}")
        
        print(
            f"Book Imbalance        : "
            f"{orderbook_imbalance:.2%}"
        )

        print("\n" + "="*40)
        print("PHASE 4")
        print("="*40)
        
        print(f"Whale Buys          : {whale_buys}")
        print(f"Whale Sells         : {whale_sells}")
        
        print(f"Whale Buy Value     : {whale_buy_value:,.2f}")
        print(f"Whale Sell Value    : {whale_sell_value:,.2f}")
        
        print(f"Cluster Score       : {cluster_score}")
        
        print(f"Absorption Score    : {absorption_score}")

        print("\n" + "="*40)
        print("PHASE 5")
        print("="*40)
        
        print(f"Market State             : {market_state}")
        
        print(
            f"Bullish Absorption Score : "
            f"{bullish_absorption_score}"
        )
        
        print(
            f"Bearish Absorption Score : "
            f"{bearish_absorption_score}"
        )

        print("\n" + "="*40)
        print("PHASE 6")
        print("="*40)
        
        print(f"Long Score       : {long_score}")
        print(f"Short Score      : {short_score}")
        
        print(
            f"Long Confidence  : "
            f"{long_confidence:.2f}%"
        )
        
        print(
            f"Short Confidence : "
            f"{short_confidence:.2f}%"
        )
        
        print(f"Signal           : {trade_signal}")
        await asyncio.sleep(5)

In [141]:
await asyncio.gather(
    trade_stream(),
    delta_engine(),
    orderbook_stream(),
    absorption_engine(),
    whale_cluster_engine(),
    market_state_engine(),
    sniper_engine(),
    dashboard()
)


PHASE 1
Price       : 0.00
Buy Volume  : 0.00
Sell Volume : 0.00
Trades      : 0

PHASE 2
Interval Buy Volume  : 0.00
Interval Sell Volume : 0.00
Buy/Sell Ratio       : 0.00
Delta       : 0.00
CVD         : 0.00

PHASE 3
Bid Liquidity         : 25.80
Ask Liquidity         : 0.58
Book Imbalance        : 95.62%

PHASE 4
Whale Buys          : 0
Whale Sells         : 0
Whale Buy Value     : 0.00
Whale Sell Value    : 0.00
Cluster Score       : 0
Absorption Score    : 0

PHASE 5
Market State             : NEUTRAL
Bullish Absorption Score : 0
Bearish Absorption Score : 0

PHASE 6
Long Score       : 20
Short Score      : 65
Long Confidence  : 23.53%
Short Confidence : 76.47%
Signal           : NO TRADE
📚 Order Book Connected
Connected to Binance

PHASE 1
Price       : 73,656.20
Buy Volume  : 26,737.20
Sell Volume : 13,258.10
Trades      : 21

PHASE 2
Interval Buy Volume  : 25,632.36
Interval Sell Volume : 13,258.10
Buy/Sell Ratio       : 0.00
Delta       : 12,374.26
CVD         : 12,374.26



CancelledError: 


PHASE 1
Price       : 73,567.70
Buy Volume  : 7,702,928.74
Sell Volume : 12,514,913.87
Trades      : 7209

PHASE 2
Interval Buy Volume  : 147.14
Interval Sell Volume : 5,738.28
Buy/Sell Ratio       : 0.00
Delta       : -5,591.15
CVD         : -4,811,985.12

PHASE 3
Bid Liquidity         : 7.78
Ask Liquidity         : 3.77
Book Imbalance        : 34.74%

PHASE 1
Price       : 73,567.70
Buy Volume  : 7,702,928.74
Sell Volume : 12,514,913.87
Trades      : 7209

PHASE 2
Interval Buy Volume  : 0.00
Interval Sell Volume : 0.00
Buy/Sell Ratio       : 0.00
Delta       : 0.00
CVD         : -4,811,985.12

PHASE 3
Bid Liquidity         : 9.40
Ask Liquidity         : 2.98
Book Imbalance        : 51.85%

PHASE 4
Whale Buys          : 7
Whale Sells         : 7
Whale Buy Value     : 907,735.90
Whale Sell Value    : 818,793.63
Cluster Score       : 14
Absorption Score    : 0

PHASE 1
Price       : 73,567.70
Buy Volume  : 7,702,928.74
Sell Volume : 12,514,913.87
Trades      : 7209

PHASE 2
Interval Bu